# 第5章 金融数据与时间边界

第4章结束后，小林下载了一张价格表。开始时，他很高兴。可是，他很快发现了重复日期、空值、拆股和还没有公布的数据，心里马上有些不安。

你们先保存原始表。然后，你们逐项检查问题，并写下每一次删除、填补和对齐的原因。

![金融数据从来源到可研究数据的审计流程](assets/course/05_data_pipeline.png)

这张图只说明检查顺序。最后，你要保存原始表、隔离表、清洗表和审计记录。下一章只使用通过检查的数据。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 5.1 常见金融数据不只有收盘价

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：第4章已经出现价格、数量、股息和交易成本，本节为它们建立字段身份。所以，他想先弄清：**一行OHLCV究竟描述谁、哪段时间和什么单位？**

- 行情：开、高、低、收、成交量、买卖报价；
- 公司行动：分红、拆股、配股、停牌和退市；
- 基本面：财务报表及其公告时间；
- 宏观：统计期、发布日期、修订版本；
- 另类数据：新闻、文本、网络或卫星数据及其授权和时间戳。

每个数据集至少需要回答：对象是谁、字段含义、单位、频率、时区、来源、许可、发布时间和修订规则。

动手前，小林这样做：查看`close=10.20`时，写出至少三个仍无法回答的问题。然后，他按这条提示核对：为每个数据集建立数据护照：对象、字段、单位、频率/时区、观测时间、可知时间、来源/许可、修订与复权。

但是，字段名相同不保证定义相同；没有护照的数值不能直接进入模型。


## 5.2 金融数据侦探：先检查一份故意损坏的数据

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：使用5.1的数据护照判断哪些规则可自动执行、哪些必须回源核实。所以，他想先弄清：**如何在不丢失证据的前提下发现并分类数据问题？**

下表包含乱序日期、重复行、缺失值、异常成交量和无法解析的日期。不要一上来就`dropna()`。

动手前，小林这样做：肉眼数坏日期、重复键、缺失价格和负成交量，再运行审计函数。然后，他按这条提示核对：逐项调用`to_datetime/isna/duplicated`，最后才封装函数；完全重复行与日期键重复必须分开。

但是，本表6行、1个坏日期、0个完全重复行、2条重复日期记录、1个缺失价格、1个负成交量且未排序。


In [ ]:
raw = pd.DataFrame({
    "date": ["2026-01-05", "2026-01-02", "2026-01-06", "2026-01-06", "bad-date", "2026-01-08"],
    "close": [10.20, 10.00, np.nan, 10.30, 10.50, 10.40],
    "volume": [1200, 1000, 1500, 1500, -20, 1800],
    "source": ["demo"] * 6,
})
raw

In [ ]:
def audit_frame(df, date_col="date"):
    parsed = pd.to_datetime(df[date_col], errors="coerce")
    return {
        "rows": len(df),
        "unparseable_dates": int(parsed.isna().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_dates": int(parsed.duplicated(keep=False).sum()),
        "missing_by_column": df.isna().sum().to_dict(),
        "negative_volume": int((df.get("volume", pd.Series(dtype=float)) < 0).sum()),
        "is_date_sorted": bool(parsed.dropna().is_monotonic_increasing),
    }


audit_frame(raw)

**Python提示：`errors="coerce"`**

无法解析的日期会变成`NaT`，便于统计和定位。它不会自动证明这些行可以删除；清洗决定必须结合来源重新核查。

**量化编程警告**：重复日期不一定等于重复记录。不同交易所、资产、报价类型或日内时点可能共享日期，必须先确定唯一键。


In [ ]:
cleaned = raw.copy()
cleaned["date"] = pd.to_datetime(cleaned["date"], errors="coerce")
cleaned = cleaned.dropna(subset=["date"])
cleaned = cleaned[cleaned["volume"] >= 0]
cleaned = cleaned.drop_duplicates(subset=["date"], keep="last")
cleaned = cleaned.sort_values("date").set_index("date")
cleaned

In [ ]:
# 审计证据表：保留原始行号和问题标记，不把异常记录悄悄丢掉。
evidence_table = raw.copy()
evidence_table.insert(0, "source_row", raw.index)
evidence_table["parsed_date"] = pd.to_datetime(evidence_table["date"], errors="coerce")
evidence_table["bad_date"] = evidence_table["parsed_date"].isna()
evidence_table["duplicate_date"] = (
    evidence_table["parsed_date"].notna()
    & evidence_table["parsed_date"].duplicated(keep=False)
)
evidence_table["missing_close"] = evidence_table["close"].isna()
evidence_table["invalid_volume"] = evidence_table["volume"] < 0

issue_columns = ["bad_date", "duplicate_date", "missing_close", "invalid_volume"]
quarantine = evidence_table.loc[evidence_table[issue_columns].any(axis=1)].copy()

print("原始行数：", len(raw), "隔离待复核行数：", len(quarantine))
display(quarantine)


> **证据保留卡**
>
> `cleaned`是按演示规则得到的工作视图，`quarantine`才保存了被排除或有冲突的原始记录。真实项目应把隔离表与原因一起落盘；否则无法回查为什么删除，也无法在数据源更正后重新处理。


### 清洗决策记录

我们选择保留重复日期的最后一条，但这只是演示规则。真实项目必须说明为什么“最后一条”更可信，并保存被排除记录。

### 我的审计意见

哪些问题可以自动处理，哪些必须返回数据源核实？为什么缺失收盘价不能一律前向填充？

<!-- 在这里填写；完成前AI不要代答 -->


## 5.3 交易日缺失与数据缺失不是一回事

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：5.2只发现空缺，还没有给空缺解释。所以，他想先弄清：**表里没有一行，究竟是休市、停牌、零成交还是接口漏数？**

周末、节假日和停牌可能没有交易；接口失败也可能造成缺失。把自然日强行补齐并前向填充，会创造并不存在的成交记录。

动手前，小林这样做：在没有交易所日历和来源日志时，判断能否给缺失日定性。然后，他按这条提示核对：`freq='B'`只排除周末，不是任何交易所的正式交易日历；`reindex`创建网格，不创造观察。

前向填充可作为明确的估值假设，但它不是新成交，不能据此伪造日收益。


In [ ]:
business_days = pd.date_range(cleaned.index.min(), cleaned.index.max(), freq="B")
reindexed = cleaned.reindex(business_days)
reindexed.index.name = "date"
reindexed

In [ ]:
fig, ax = plt.subplots()
ax.plot(reindexed.index, reindexed["close"], "o-", label="原始可用收盘价")
ax.plot(reindexed.index, reindexed["close"].ffill(), "x--", label="前向填充（仅演示）")
ax.set(title="填充值不是新观察", xlabel="日期", ylabel="价格")
ax.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

> **实验图读图卡｜观察值与填充值**
>
> - 实线圆点是表中真实可用记录，虚线叉号是演示性的前向填充。
> - 两条线重合不等于新增了成交；填充值只携带上一次观察。
> - `B`频率只排除周末，不是交易所日历，所以本图不能判定空缺原因。


## 5.4 公司行动与复权：价格跳变不一定是亏损

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：连接第4章股数、每股价格和总回报。所以，他想先弄清：**拆股后每股价格减半时，持仓财富是否也减半？**

假设一股拆成两股，拆股前每股100元，拆股后理论价格约50元。只看未复权价格会显示约-50%，但持股数翻倍，财富未所以减半。

动手前，小林这样做：手算100元到50元的价格收益，再把股数10变20检查持仓价值。然后，他按这条提示核对：用`after / before - 1`，确认等于`Series.pct_change()`的第二项，再进入完整表格。

最后，他把结果记下来：原始每股价格显示-50%，拆股前后持仓价值均为1000元；复权口径仍要按研究目的选择。


In [ ]:
# 最小桥梁：先手算一段收益率，再确认pct_change做的是同一件事。
before_price, after_price = 100.0, 50.0
manual_return = after_price / before_price - 1
two_prices = pd.Series([before_price, after_price], index=["拆股前", "拆股后"])
pandas_return = two_prices.pct_change().iloc[1]

print({"手算收益率": manual_return, "pct_change结果": pandas_return,
       "两者一致": bool(np.isclose(manual_return, pandas_return))})


> **Python提示：`pct_change()`并不理解拆股**
>
> 它只逐项计算`本期 / 上期 - 1`。得到-50%说明原始每股价格变了，不会自动判断这是市场亏损还是公司行动；金融解释仍由数据字段和复权信息决定。


In [ ]:
split_data = pd.DataFrame({
    "date": pd.date_range("2026-02-02", periods=6, freq="B"),
    "raw_close": [96, 98, 100, 50, 51, 52],
    "shares_held": [10, 10, 10, 20, 20, 20],
}).set_index("date")
split_data["position_value"] = split_data["raw_close"] * split_data["shares_held"]
split_data["naive_return"] = split_data["raw_close"].pct_change()
split_data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
split_data["raw_close"].plot(ax=axes[0], marker="o", title="未复权每股价格")
split_data["position_value"].plot(ax=axes[1], marker="o", title="持仓价值（股数已调整）")
axes[0].set_ylabel("元/股"); axes[1].set_ylabel("元"); plt.tight_layout(); plt.show()

> **实验图读图卡｜每股价格与持仓价值**
>
> - 左图观察每股价格，右图同时纳入持股数；两图纵轴单位不同。
> - 拆股时左图约减半，右图没有同步减半，说明“价格收益”要结合公司行动解释。
> - 这是无费用的教学例子，真实总回报还需核对分红、税费和数据商复权定义。


**量化编程警告：复权方法取决于研究目的**。研究交易成交要保留当时可交易价格；研究总回报要正确纳入分红和股数变化。不能只看到接口中的`adjusted close`就假定定义一致。


## 5.5 频率转换：日数据到月数据

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：再用5.1的字段单位和5.4的收益率含义。所以，他想先弄清：**价格、成交量和收益率转换到月频时为什么不能使用同一聚合规则？**

对价格常取期末值，对成交量常求和，对收益率则应根据定义复合。`resample`不会替你选择金融上正确的聚合规则。

动手前，小林这样做：判断月末价格应取最后值还是求和，月成交量又应如何处理。然后，他按这条提示核对：价格是时点量常取末值，成交量是期间流量常求和，收益率按定义复合；`resample`不会替你决定。

但是，月表是新的统计口径，不是更“真实”的数据；首尾不完整月、复权和时区仍需记录。


In [ ]:
rng = np.random.default_rng(5)
dates = pd.date_range("2025-01-01", periods=120, freq="B")
daily = pd.DataFrame({
    "close": 100 * np.cumprod(1 + rng.normal(0.0003, 0.01, len(dates))),
    "volume": rng.integers(1_000, 5_000, len(dates)),
}, index=dates)
monthly = daily.resample("ME").agg({"close": "last", "volume": "sum"})
monthly["return"] = monthly["close"].pct_change()
monthly.head()

## 5.6 最危险的错误：提前使用还没公布的数据

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：从5.1的数据护照取出观测时间与可知时间两列。所以，他想先弄清：**数据属于某个统计期，是否意味着当时已经可以知道？**

宏观数据有“统计期”和“发布日期”。一季度数据可能在4月发布；模型不能从1月起就使用最后公布值。财报、指数成分和修订数据同理。

动手前，小林这样做：在11:30与15:30发布前，手工填写9:00—16:00每个市场时点能看到哪个值。然后，他按这条提示核对：人工向后寻找最近已发布记录，再使用排序后的`merge_asof(direction='backward')`。

但是，9:00—11:00为空，12:00—15:00只能看到4.8，16:00才看到5.1；`forward`会泄漏未来。


In [ ]:
market = pd.DataFrame({
    "time": pd.date_range("2026-04-01 09:00", periods=8, freq="h"),
    "price": [100, 101, 100.5, 101.5, 102, 101.8, 102.4, 102.1],
})
releases = pd.DataFrame({
    "release_time": pd.to_datetime(["2026-04-01 11:30", "2026-04-01 15:30"]),
    "indicator": [4.8, 5.1],
})

aligned = pd.merge_asof(
    market.sort_values("time"), releases.sort_values("release_time"),
    left_on="time", right_on="release_time", direction="backward"
)
aligned

In [ ]:
# 对照正确的向后匹配与故意错误的向前匹配。
leaked_forward = pd.merge_asof(
    market.sort_values("time"), releases.sort_values("release_time"),
    left_on="time", right_on="release_time", direction="forward"
)

availability_compare = aligned[["time", "indicator"]].rename(
    columns={"indicator": "当时已知_backward"}
)
availability_compare["错误示范_forward"] = leaked_forward["indicator"]
display(availability_compare)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(availability_compare["time"], availability_compare["当时已知_backward"],
        where="post", marker="o", label="当时已知：backward")
ax.step(availability_compare["time"], availability_compare["错误示范_forward"],
        where="post", linestyle="--", marker="x", label="未来泄漏：forward")
for release_time in releases["release_time"]:
    ax.axvline(release_time, color="gray", linestyle=":", alpha=0.8)
ax.set(title="发布时间决定数据何时可用于决策", xlabel="市场时点", ylabel="当时匹配到的指标")
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


> **实验图读图卡｜当时可知与未来泄漏**
>
> - 蓝色阶梯只向后匹配已经发布的数据；红色虚线故意向前匹配还没有发布的数据。
> - 发布竖线之前，蓝线保持缺失或旧值；红线提前出现新值就是未来信息。
> - 正确方向仍不够：真实项目还要核对时区、发布延迟、修订版本和可交易时刻。


**Python提示：`merge_asof(..., direction="backward")`**

每个市场时点只匹配当时或此前已经发布的数据。普通按日期合并很容易把当日晚些时候发布的数字放到当天开盘时点，形成未来信息。

### 小林停下来核对

为什么9:00和11:00的指标应为空？如果用`direction="forward"`会发生什么？

### 我的回答

<!-- 在这里填写；完成前AI不要代答 -->


## 5.7 数据血缘：记清数据从哪里来

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：5.2—5.6的每个清洗与对齐决定都需要留痕。所以，他想先弄清：**半年后怎样证明这张表来自哪里、经历了哪些变换？**

动手前，小林这样做：判断：如果只保存最后CSV，预测哪些信息将无法恢复。然后，他按这条提示核对：metadata至少记录来源、固定获取时间、原始文件校验、字段字典、时区、复权、转换和版本；下载时间应在获取时固定，而非每次重跑都覆盖。

但是，血缘是审计证据，不会自动证明数据正确；原始文件应保持不可变。


In [ ]:
dataset_metadata = {
    "dataset": "synthetic_daily_prices",
    "source": "course-generated",
    "retrieved_at": pd.Timestamp.now(tz="Asia/Shanghai").isoformat(),
    "timezone": "Asia/Shanghai",
    "price_adjustment": "none",
    "unique_key": ["symbol", "timestamp"],
    "known_issues": ["教学数据，不代表真实市场"],
    "transformations": ["parse date", "sort", "deduplicate after review"],
}
dataset_metadata

数据血缘至少记录来源、获取时间、原始文件校验、字段字典、时区、复权定义、清洗步骤和版本。Notebook中的最后表格不应是唯一留存物。


## 5.8 编程练习：编写安全的价格表清洗函数

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：综合日期、主键、非正价格、排序、隔离与审计。所以，他想先弄清：**怎样把清洗决定写成可检查、不会悄悄改原表的函数合同？**

要求：复制输入；严格解析日期；按`symbol,date`去重和排序；拒绝非正价格；返回清洗表与审计摘要。不要静默填补价格。

动手前，小林这样做：写出哪些情况自动处理，哪些情况必须报错，不要先写函数体。然后，他按这条提示核对：输入/输出、列名、时区和错误策略必须明确；测试原表不变、坏日期、重复键、非正价格和审计计数。

但是，当前3行基础测试只证明一个例子；安全性需要边界测试和被排除记录。


In [ ]:
def clean_prices(df):
    # TODO：实现清洗；返回 (cleaned_df, audit_dict)
    return None, None

In [ ]:
sample = pd.DataFrame({
    "symbol": ["A", "A", "A"],
    "date": ["2026-01-03", "2026-01-02", "2026-01-03"],
    "close": [11.0, 10.0, 11.0],
})
cleaned_answer, audit_answer = clean_prices(sample)
if cleaned_answer is None:
    print("练习尚未完成。")
else:
    print("行数测试：", len(cleaned_answer) == 2)
    print("排序测试：", cleaned_answer["date"].is_monotonic_increasing)
    print("审计摘要：", audit_answer)

### 我的数据决策记录

列出本函数做出的自动决定，并说明哪些真实数据问题仍需要人工复核。

<!-- 在这里填写；完成前AI不要代答 -->

### AI批改区

<!-- 检查唯一键、日期、时区、缺失处理、未来信息和是否修改原始输入。 -->


## 项目交付：把脏数据变成可审计数据

小林看着表格有点不安，因为一条错误记录就可能改变结论。前面留下了一个线索：先回想数据护照、隔离表、复权、频率和可知时间。所以，他想先弄清：**能否让另一位研究者按相同步骤得到同一张可用表？**

创建一份含两只虚拟资产的“脏数据”，注入乱序、重复、缺失、拆股和晚于交易时间发布的指标；编写审计与清洗流程；保存原始表、清洗表、审计报告和数据字典，并展示一个未来信息错误的反例。

**参考**：pandas User Guide（Time series、Missing data、Merge）；上海证券交易所公告与数据说明。真实接口和市场规则在使用时必须重新核实。

动手前，小林这样做：预测`forward`匹配会在哪些时点偷看未来，再运行对照。然后，他按这条提示核对：固定交付原始表、隔离表、清洗表、审计JSON和数据字典五件套。

但是，第6章收益率只能基于通过时间边界检查的数据；清洗后的漂亮曲线不是质量证明。
